# Joint on-device ASR training notebook for Central Puebla Nahuatl and Seri

This notebook downloads the latest open Common Voice scripted-speech datasets for **Central Puebla Nahuatl (`ncx`)** and **Seri (`sei`)**, builds a multilingual training set, fine-tunes a **small Wav2Vec2/XLS-R CTC** speech-to-text model, evaluates it, and exports an **ONNX + INT8** version for on-device inference.

Why this setup?

- It works well for **low-resource ASR**.
- It lets us build a **custom character vocabulary** directly from the Nahuatl and Seri transcripts.
- It has a clean export path to **ONNX Runtime** for edge/mobile deployment.
- You can keep the notebook **joint** (`["ncx", "sei"]`) or switch to **one language at a time** (`["ncx"]` or `["sei"]`).

Before you run it:

1. Create a Mozilla Data Collective account.
2. Open the two dataset pages in your browser and **agree to each dataset's terms**.
3. Create an **API key** in Mozilla Data Collective and set `MDC_API_KEY`.
4. If you already downloaded the `.tar.gz` files manually, you can skip the API and point `LOCAL_ARCHIVES` at your local files.

The notebook also accepts **extra community-curated recordings** through simple CSV/TSV manifests so you can blend Common Voice with your own archival speech.

## 1) Install dependencies

If you are in Colab or Ubuntu and MP3 decoding fails, uncomment the `ffmpeg` line first.

In [ ]:

# Optional system dependency for MP3 decoding on Linux / Colab:
# !apt-get -y install ffmpeg

%pip -q install -U   "datacollective>=0.4.5"   "datasets[audio]>=2.20.0"   "transformers>=4.46.0"   "accelerate>=0.34.0"   "evaluate>=0.4.2"   "jiwer>=3.0.5"   "librosa>=0.10.2"   "soundfile>=0.12.1"   "optimum[onnxruntime]>=1.23.0"   wandb pandas ipywidgets

In [ ]:
import os
os.environ["MDC_API_KEY"] = "bd2bdbcaa2e820d314a828415bc4c0afde802e259a9392102986a77a2d369c5a"

## 2) Imports

In [ ]:

import csv
import hashlib
import json
import os
import platform
import random
import re
import tarfile
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import evaluate
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display
from datasets import Audio as HFAudio
from datasets import Dataset, DatasetDict
from transformers import (
    AutoConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
)

pd.set_option("display.max_colwidth", 120)

## 3) Configuration

The defaults below train **one joint model** for both languages.

Change these if needed:

- `SELECTED_LANGS = ["ncx"]` for Central Puebla Nahuatl only
- `SELECTED_LANGS = ["sei"]` for Seri only
- `LOCAL_ARCHIVES[...] = "/path/to/common-voice-....tar.gz"` if you already downloaded the files
- `EXTRA_MANIFESTS = ["/path/to/my_train_manifest.csv"]` to add your own recordings

In [ ]:

# Project paths
PROJECT_NAME = "ncx-sei-joint-asr"
WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR = WORK_DIR / "mdc_data"
ARTIFACT_DIR = WORK_DIR / "artifacts" / PROJECT_NAME
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Languages and Mozilla Data Collective dataset IDs
LANGUAGE_CONFIG = {
    "ncx": {
        "name": "Central Puebla Nahuatl",
        "dataset_id": "cmn2c9fhc01beo1074z081p8t",
    },
    "sei": {
        "name": "Seri",
        "dataset_id": "cmn2akuj101brmm0750ppoekz",
    },
}
SELECTED_LANGS = ["ncx", "sei"]

# Optional: use manually downloaded .tar.gz archives or extracted folders instead of the MDC API
LOCAL_ARCHIVES = {
    "ncx": None,  # e.g. "/absolute/path/to/common-voice-scripted-speech-25-0-centra-....tar.gz"
    "sei": None,  # e.g. "/absolute/path/to/common-voice-scripted-speech-25-0-seri-....tar.gz"
}

# Optional: add your own curated data manifests (CSV/TSV) with columns:
#   audio_path, text, language
# Optional columns:
#   split, client_id, path
EXTRA_MANIFESTS: List[str] = []

# Split strategy
# "official_plus_extra_train": use dev/test from Common Voice and train on all remaining validated clips
# "official_only": use train/dev/test exactly as provided
# "speaker_hash": ignore official splits and create deterministic speaker-based splits from validated.tsv
SPLIT_STRATEGY = "official_plus_extra_train"

# Base model candidates: first available model wins
# The 100M checkpoint is the preferred "small" option for this notebook.
BASE_MODEL_CANDIDATES = [
    "facebook/wav2vec2-xls-r-100m",
    "facebook/wav2vec2-xls-r-300m",
]

# Transcript normalization
LOWERCASE = True
KEEP_DIGITS = True
KEEP_APOSTROPHE = True
MIN_LABEL_CHARS = 1
MAX_AUDIO_SECONDS = 20.0

# Training hyperparameters (safe defaults for a single GPU; lower batch size if you hit OOM)
SEED = 42
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 3e-4
NUM_TRAIN_EPOCHS = 15
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 3
SAVE_TOTAL_LIMIT = 2

# Processing
NUM_PROC = 1 if platform.system().lower().startswith("win") else max(1, min(4, os.cpu_count() or 1))

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("WORK_DIR     :", WORK_DIR) 
print("DATA_DIR     :", DATA_DIR)
print("ARTIFACT_DIR :", ARTIFACT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device   :", torch.cuda.get_device_name(0))

## 4) Helpers for download, extraction, parsing, normalization, and split construction

In [ ]:

def _find_common_voice_root(root: Path) -> Optional[Path]:
    root = Path(root)
    if not root.exists():
        return None

    # Direct hit
    if (root / "clips").is_dir() and any((root / name).exists() for name in ["validated.tsv", "train.tsv", "dev.tsv", "test.tsv"]):
        return root

    # Search recursively
    candidates = []
    for clips_dir in root.rglob("clips"):
        parent = clips_dir.parent
        if any((parent / name).exists() for name in ["validated.tsv", "train.tsv", "dev.tsv", "test.tsv"]):
            candidates.append(parent)

    if not candidates:
        return None

    # Prefer the shallowest match
    return sorted(candidates, key=lambda p: len(p.parts))[0]


def _safe_extract_tar_gz(archive_path: Path, extract_dir: Path) -> Path:
    archive_path = Path(archive_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, "r:gz") as tf:
        tf.extractall(extract_dir)
    return extract_dir


def _materialize_dataset(language_code: str, cfg: Dict[str, str]) -> Path:
    local_override = LOCAL_ARCHIVES.get(language_code)
    if local_override:
        candidate = Path(local_override).expanduser().resolve()
    else:
        if not os.environ.get("MDC_API_KEY"):
            raise RuntimeError(
                "MDC_API_KEY is not set and LOCAL_ARCHIVES is empty. "
                "Either export MDC_API_KEY or point LOCAL_ARCHIVES at the downloaded dataset."
            )
        from datacollective import download_dataset

        print(f"Downloading {cfg['name']} from Mozilla Data Collective ...")
        candidate = Path(download_dataset(cfg["dataset_id"])).expanduser().resolve()

    # If already extracted / a dataset folder, return it
    cv_root = _find_common_voice_root(candidate)
    if cv_root is not None:
        return cv_root

    # If the path is a directory containing archives, pick the first .tar.gz
    archive_path = None
    if candidate.is_dir():
        archives = sorted(candidate.rglob("*.tar.gz"))
        if archives:
            archive_path = archives[0]
    elif candidate.is_file() and candidate.suffixes[-2:] == [".tar", ".gz"]:
        archive_path = candidate

    if archive_path is None:
        raise FileNotFoundError(
            f"Could not find a Common Voice archive or extracted dataset under: {candidate}"
        )

    extract_dir = DATA_DIR / language_code / "extracted"
    if _find_common_voice_root(extract_dir) is None:
        print(f"Extracting {archive_path.name} ...")
        _safe_extract_tar_gz(archive_path, extract_dir)

    cv_root = _find_common_voice_root(extract_dir)
    if cv_root is None:
        raise FileNotFoundError(f"Extraction finished but no Common Voice root was found under: {extract_dir}")

    return cv_root


def _read_tsv(path: Path) -> Optional[pd.DataFrame]:
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(
        path,
        sep="\t",
        quoting=csv.QUOTE_NONE,
        keep_default_na=False,
        na_filter=False,
        low_memory=False,
    )


def _text_column(df: pd.DataFrame) -> str:
    if "text" in df.columns:
        return "text"
    if "sentence" in df.columns:
        return "sentence"
    raise KeyError(f"Could not find a transcript column in dataframe columns: {list(df.columns)}")


def _prepare_common_voice_frame(df: pd.DataFrame, cv_root: Path, language_code: str, split_name: str) -> pd.DataFrame:
    df = df.copy()
    text_col = _text_column(df)
    clips_dir = cv_root / "clips"

    if "client_id" not in df.columns:
        df["client_id"] = ""

    df["audio_path"] = df["path"].astype(str).map(lambda p: str((clips_dir / p).resolve()))
    df["language"] = language_code
    df["split"] = split_name
    df["text"] = df[text_col].astype(str)

    keep_cols = ["audio_path", "text", "client_id", "language", "split", "path"]
    for col in keep_cols:
        if col not in df.columns:
            df[col] = ""
    df = df[keep_cols].copy()

    # Keep only rows where the audio file is present
    df = df[df["audio_path"].map(lambda p: Path(p).exists())].copy()
    df = df.drop_duplicates(subset=["language", "path", "split"]).reset_index(drop=True)
    return df


def _speaker_hash_split(df: pd.DataFrame, dev_ratio: float = 0.1, test_ratio: float = 0.1) -> pd.DataFrame:
    df = df.copy()

    speaker = df["client_id"].astype(str).replace("", np.nan)
    speaker = speaker.fillna(df["path"].astype(str).map(lambda x: Path(x).stem))
    df["speaker_key"] = df["language"].astype(str) + "::" + speaker.astype(str)

    splits = []
    for key in df["speaker_key"].tolist():
        h = int(hashlib.sha1(key.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
        if h < test_ratio:
            splits.append("test")
        elif h < test_ratio + dev_ratio:
            splits.append("validation")
        else:
            splits.append("train")
    df["split"] = splits
    return df


def _build_language_dataframe(cv_root: Path, language_code: str, split_strategy: str = "official_plus_extra_train") -> pd.DataFrame:
    validated = _read_tsv(cv_root / "validated.tsv")
    train_tsv = _read_tsv(cv_root / "train.tsv")
    dev_tsv = _read_tsv(cv_root / "dev.tsv")
    test_tsv = _read_tsv(cv_root / "test.tsv")

    if split_strategy == "official_plus_extra_train" and validated is not None and dev_tsv is not None and test_tsv is not None:
        held_out = set(dev_tsv["path"].astype(str)) | set(test_tsv["path"].astype(str))
        train_source = validated[~validated["path"].astype(str).isin(held_out)].copy()

        frames = [
            _prepare_common_voice_frame(train_source, cv_root, language_code, "train"),
            _prepare_common_voice_frame(dev_tsv, cv_root, language_code, "validation"),
            _prepare_common_voice_frame(test_tsv, cv_root, language_code, "test"),
        ]
        return pd.concat(frames, ignore_index=True)

    if split_strategy == "official_only" and train_tsv is not None and dev_tsv is not None and test_tsv is not None:
        frames = [
            _prepare_common_voice_frame(train_tsv, cv_root, language_code, "train"),
            _prepare_common_voice_frame(dev_tsv, cv_root, language_code, "validation"),
            _prepare_common_voice_frame(test_tsv, cv_root, language_code, "test"),
        ]
        return pd.concat(frames, ignore_index=True)

    if validated is not None:
        frame = _prepare_common_voice_frame(validated, cv_root, language_code, "train")
        frame = _speaker_hash_split(frame, dev_ratio=0.1, test_ratio=0.1)
        return frame.reset_index(drop=True)

    raise FileNotFoundError(
        f"Could not build splits under {cv_root}. Expected validated.tsv or train/dev/test TSVs."
    )


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).strip()

    if LOWERCASE:
        text = text.lower()

    apostrophe_map = {
        "’": "'",
        "ʻ": "'",
        "ʼ": "'",
        "`": "'",
        "´": "'",
        "ꞌ": "'",
    }
    for src, dst in apostrophe_map.items():
        text = text.replace(src, dst)

    cleaned = []
    for ch in text:
        cat = unicodedata.category(ch)
        keep = False

        if ch == " ":
            keep = True
        elif KEEP_APOSTROPHE and ch == "'":
            keep = True
        elif cat.startswith("L") or cat.startswith("M"):
            keep = True
        elif KEEP_DIGITS and cat == "Nd":
            keep = True

        cleaned.append(ch if keep else " ")

    text = "".join(cleaned)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_extra_manifest(path_like: str) -> pd.DataFrame:
    path = Path(path_like).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(path)

    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    df = pd.read_csv(path, sep=sep, keep_default_na=False, na_filter=False)

    required = {"audio_path", "text", "language"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest {path} is missing required columns: {missing}")

    if "split" not in df.columns:
        df["split"] = "train"
    if "client_id" not in df.columns:
        df["client_id"] = ""
    if "path" not in df.columns:
        df["path"] = df["audio_path"].astype(str).map(lambda p: Path(p).name)

    df = df[["audio_path", "text", "client_id", "language", "split", "path"]].copy()
    df["audio_path"] = df["audio_path"].astype(str).map(lambda p: str(Path(p).expanduser().resolve()))
    df = df[df["audio_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)
    return df


def resolve_base_model(candidates: Iterable[str]) -> str:
    errors = {}
    for model_id in candidates:
        try:
            _ = AutoConfig.from_pretrained(model_id)
            return model_id
        except Exception as exc:
            errors[model_id] = repr(exc)
    raise RuntimeError(f"None of the base model candidates could be loaded: {errors}")


def dataframe_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        df[["audio_path", "text_norm", "language", "path"]].rename(columns={"text_norm": "text"}),
        preserve_index=False,
    )
    ds = ds.cast_column("audio_path", HFAudio(sampling_rate=16000))
    ds = ds.rename_column("audio_path", "audio")
    return ds

## 5) Download / extract the datasets and build the working dataframe

If you already exported your API key in the shell, this cell will use it automatically.

Examples:

- Linux/macOS: `export MDC_API_KEY=...`
- Windows PowerShell: `$env:MDC_API_KEY="..."`

If `MDC_API_KEY` is not set, the cell will still work **if** you filled `LOCAL_ARCHIVES`.

In [ ]:

if not os.environ.get("MDC_API_KEY") and all(v is None for v in LOCAL_ARCHIVES.values()):
    print(
        "MDC_API_KEY is not set. "
        "Set it before running this cell, or point LOCAL_ARCHIVES at your manually-downloaded dataset archives."
    )

dataset_roots = {}
language_frames = []

for lang in SELECTED_LANGS:
    cfg = LANGUAGE_CONFIG[lang]
    cv_root = _materialize_dataset(lang, cfg)
    dataset_roots[lang] = cv_root
    print(f"{lang}: {cv_root}")

    lang_df = _build_language_dataframe(cv_root, lang, split_strategy=SPLIT_STRATEGY)
    language_frames.append(lang_df)

metadata_df = pd.concat(language_frames, ignore_index=True)

# Optional user-provided manifests
for manifest in EXTRA_MANIFESTS:
    extra_df = load_extra_manifest(manifest)
    metadata_df = pd.concat([metadata_df, extra_df], ignore_index=True)

metadata_df["text_norm"] = metadata_df["text"].map(normalize_text)
metadata_df = metadata_df[metadata_df["text_norm"].str.len() >= MIN_LABEL_CHARS].copy()
metadata_df = metadata_df.reset_index(drop=True)

summary = (
    metadata_df.groupby(["language", "split"], dropna=False)
    .agg(
        clips=("audio_path", "count"),
        speakers=("client_id", lambda x: pd.Series(x).replace("", np.nan).nunique()),
        unique_prompts=("text_norm", "nunique"),
    )
    .reset_index()
    .sort_values(["language", "split"])
)

display(summary)
display(metadata_df.head(10))

## 6) Inspect the normalized character inventory

This is the vocabulary the CTC head will learn to emit.

In [ ]:

all_train_text = " ".join(metadata_df.loc[metadata_df["split"] == "train", "text_norm"].tolist())
vocab_chars = sorted(set(all_train_text))

if "|" in vocab_chars:
    raise ValueError("The pipe character '|' already exists in the transcripts. Change the word delimiter token before continuing.")

if " " not in vocab_chars:
    vocab_chars.append(" ")

vocab_dict = {char: idx for idx, char in enumerate(vocab_chars)}
vocab_dict["|"] = vocab_dict.pop(" ")
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

PROCESSOR_INIT_DIR = ARTIFACT_DIR / "processor_init"
PROCESSOR_INIT_DIR.mkdir(parents=True, exist_ok=True)

with open(PROCESSOR_INIT_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f"Vocabulary size: {len(vocab_dict)}")
print("Vocabulary:")
print(" ".join(vocab_dict.keys()))

## 7) Create the tokenizer / processor and Hugging Face datasets

In [ ]:

BASE_MODEL = resolve_base_model(BASE_MODEL_CANDIDATES)
print("Using base model:", BASE_MODEL)

tokenizer = Wav2Vec2CTCTokenizer(
    str(PROCESSOR_INIT_DIR / "vocab.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
processor.save_pretrained(PROCESSOR_INIT_DIR)

raw_datasets = DatasetDict(
    {
        split: dataframe_to_hf_dataset(metadata_df[metadata_df["split"] == split].reset_index(drop=True))
        for split in ["train", "validation", "test"]
        if (metadata_df["split"] == split).any()
    }
)

raw_datasets

## 8) Convert audio + text into model inputs

This step decodes audio, resamples it to **16 kHz**, computes input values, and tokenizes the normalized transcript.

In [ ]:

def prepare_batch(batch: Dict) -> Dict:
    audio = batch["audio"]
    processed_audio = processor(audio["array"], sampling_rate=audio["sampling_rate"])

    batch["input_values"] = processed_audio.input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


encoded_datasets = raw_datasets.map(
    prepare_batch,
    remove_columns=["audio"],
    num_proc=NUM_PROC,
    desc="Preparing audio and labels",
)

encoded_datasets = encoded_datasets.filter(
    lambda input_length, labels: (input_length > 0) and (input_length < int(MAX_AUDIO_SECONDS * 16000)) and (len(labels) > 0),
    input_columns=["input_length", "labels"],
    desc="Filtering empty / very long examples",
)

for split in encoded_datasets:
    print(split, len(encoded_datasets[split]))

## 9) Data collator and evaluation metrics

In [ ]:

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: bool = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor)


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

## 10) Load the pretrained XLS-R checkpoint and configure the CTC head

In [ ]:

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ignore_mismatched_sizes=True,
)

# Recommended for low-resource ASR
model.freeze_feature_encoder()
model.config.ctc_zero_infinity = True
model.config.use_cache = False

print("Number of output tokens:", model.lm_head.out_features)

## 11) Training arguments

These defaults are conservative and aimed at a typical single-GPU fine-tune.

If you hit CUDA OOM:

- lower `PER_DEVICE_TRAIN_BATCH_SIZE` from 4 to 2
- increase `GRADIENT_ACCUMULATION_STEPS`
- keep the rest unchanged

In [ ]:
import wandb
wandb.init(project=PROJECT_NAME, name=PROJECT_NAME + "-run1")


CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
FINAL_MODEL_DIR = ARTIFACT_DIR / "final-model"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    group_by_length=True,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    save_total_limit=SAVE_TOTAL_LIMIT,
    dataloader_num_workers=0 if platform.system().lower().startswith("win") else 2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="wandb",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

training_args

## 12) Train

In [ ]:

train_result = trainer.train()
train_metrics = train_result.metrics

trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

with open(ARTIFACT_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)

train_metrics

## 13) Evaluate on the test set and report per-language scores

In [ ]:

test_metrics = trainer.evaluate(encoded_datasets["test"])
with open(ARTIFACT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

test_pred = trainer.predict(encoded_datasets["test"])
pred_ids = np.argmax(test_pred.predictions, axis=-1)

label_ids = test_pred.label_ids.copy()
label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

pred_str = processor.batch_decode(pred_ids)
label_str = processor.batch_decode(label_ids, group_tokens=False)

pred_df = pd.DataFrame(
    {
        "language": encoded_datasets["test"]["language"],
        "path": encoded_datasets["test"]["path"],
        "reference": label_str,
        "prediction": pred_str,
    }
)

def score_subset(df: pd.DataFrame) -> Dict[str, float]:
    refs = df["reference"].tolist()
    hyps = df["prediction"].tolist()
    return {
        "wer": float(wer_metric.compute(references=refs, predictions=hyps)),
        "cer": float(cer_metric.compute(references=refs, predictions=hyps)),
        "clips": int(len(df)),
    }

language_breakdown = (
    pd.DataFrame(
        [
            {"language": lang, **score_subset(sub_df)}
            for lang, sub_df in pred_df.groupby("language", sort=True)
        ]
    )
    .sort_values("language")
    .reset_index(drop=True)
)

pred_df.to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False)
language_breakdown.to_csv(ARTIFACT_DIR / "language_breakdown.csv", index=False)

display(pd.DataFrame([test_metrics]))
display(language_breakdown)
display(pred_df.sample(min(12, len(pred_df)), random_state=SEED))

## 14) Listen to a sample prediction

In [ ]:

sample_idx = 0
sample_audio = raw_datasets["test"][sample_idx]["audio"]
display(Audio(sample_audio["array"], rate=sample_audio["sampling_rate"]))
print("Language :", raw_datasets["test"][sample_idx]["language"])
print("Reference:", pred_df.iloc[sample_idx]["reference"])
print("Predicted:", pred_df.iloc[sample_idx]["prediction"])

## 15) Export to ONNX and create a dynamic INT8 quantized version

This is the easiest path from the fine-tuned PyTorch checkpoint to an on-device-friendly runtime.

The exported files are saved to:

- `artifacts/<project>/onnx`
- `artifacts/<project>/onnx-int8`

In [ ]:

from optimum.onnxruntime import ORTModelForCTC, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ONNX_DIR = ARTIFACT_DIR / "onnx"
QUANT_DIR = ARTIFACT_DIR / "onnx-int8"
ONNX_DIR.mkdir(parents=True, exist_ok=True)
QUANT_DIR.mkdir(parents=True, exist_ok=True)

ort_model = ORTModelForCTC.from_pretrained(str(FINAL_MODEL_DIR), export=True)
ort_model.save_pretrained(str(ONNX_DIR))
processor.save_pretrained(str(ONNX_DIR))

arch = platform.machine().lower()
if arch in {"arm64", "aarch64"}:
    qconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=False)
else:
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

quantizer = ORTQuantizer.from_pretrained(str(ONNX_DIR))
_ = quantizer.quantize(
    save_dir=str(QUANT_DIR),
    quantization_config=qconfig,
)
processor.save_pretrained(str(QUANT_DIR))

print("Saved ONNX model to      :", ONNX_DIR)
print("Saved quantized model to :", QUANT_DIR)

## 16) Quick inference test with the quantized ONNX model

In [ ]:

ort_processor = Wav2Vec2Processor.from_pretrained(str(QUANT_DIR))
ort_model_q = ORTModelForCTC.from_pretrained(str(QUANT_DIR))

sample_audio = raw_datasets["test"][0]["audio"]
ort_inputs = ort_processor(
    sample_audio["array"],
    sampling_rate=sample_audio["sampling_rate"],
    return_tensors="pt",
)

ort_logits = ort_model_q(**ort_inputs).logits
if isinstance(ort_logits, torch.Tensor):
    ort_pred_ids = torch.argmax(ort_logits, dim=-1)
else:
    ort_pred_ids = np.argmax(ort_logits, axis=-1)

ort_text = ort_processor.batch_decode(ort_pred_ids)[0]
print("Quantized ONNX prediction:", ort_text)

## 17) Save a lightweight metadata file for archiving / deployment

This makes it easier to remember exactly what was trained, on which data, and with which normalization rules.

In [ ]:

model_metadata = {
    "project_name": PROJECT_NAME,
    "languages": {lang: LANGUAGE_CONFIG[lang]["name"] for lang in SELECTED_LANGS},
    "dataset_ids": {lang: LANGUAGE_CONFIG[lang]["dataset_id"] for lang in SELECTED_LANGS},
    "split_strategy": SPLIT_STRATEGY,
    "base_model": BASE_MODEL,
    "normalization": {
        "lowercase": LOWERCASE,
        "keep_digits": KEEP_DIGITS,
        "keep_apostrophe": KEEP_APOSTROPHE,
        "max_audio_seconds": MAX_AUDIO_SECONDS,
    },
    "paths": {
        "final_model_dir": str(FINAL_MODEL_DIR),
        "onnx_dir": str(ONNX_DIR),
        "onnx_int8_dir": str(QUANT_DIR),
    },
}

with open(ARTIFACT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, ensure_ascii=False, indent=2)

model_metadata

## 18) Practical next steps

1. **Add spontaneous / conversational recordings.** Common Voice is read speech, so a second round of fine-tuning on your own archival audio will matter a lot.
2. **Keep the same normalization.** Train, eval, and deployment should all use the exact same transcript cleaning rules.
3. **Track language-specific metrics.** Keep separate WER/CER for `ncx` and `sei` so one language does not silently regress.
4. **Distill later if needed.** Once the joint model is good enough, you can distill it into an even smaller student for mobile.
5. **Package with ONNX Runtime Mobile.** The exported INT8 checkpoint is the best starting point for Android/iOS or edge CPU inference.

## References used to build this notebook

- Mozilla Data Collective — Common Voice Scripted Speech 25.0, Central Puebla Nahuatl (`ncx`)
- Mozilla Data Collective — Common Voice Scripted Speech 25.0, Seri (`sei`)
- Mozilla Data Collective Python SDK (`datacollective`)
- Hugging Face — Fine-tuning XLS-R for multilingual ASR
- Hugging Face Optimum ONNX — `ORTModelForCTC` and ONNX quantization
- Meta / Hugging Face model cards — XLS-R pretrained checkpoints